In [0]:
#  Cell 1: Widgets — accept params from master pipeline
dbutils.widgets.text("region", "All", "Region Filter")
dbutils.widgets.text("start_date", "2024-01-01", "Start Date")
dbutils.widgets.text("end_date", "2024-01-01","End Date")

region = dbutils.widgets.get("region")
start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")

print(f'Running: region={region}, {start_date} → {end_date}')

In [0]:
%run ../00_setup/Config

In [0]:
%run ../utils/Common_functions

In [0]:
#  Cell 3: Re-read raw data 

from pyspark.sql.functions import col, to_date, round, when

df = spark.read.option("header","true").option("inferschema","true").csv(CSV_PATH)

df.show()

In [0]:
#  Cell 4: Cast types and clean 
df_clean = df \
.withColumn('sale_date', to_date(col('sale_date'), 'yyyy-MM-dd')) \
.withColumn('expiry_date', to_date(col('expiry_date'), 'yyyy-MM-dd')) \
.withColumn('units_sold', col('units_sold').cast('integer')) \
.withColumn('unit_price', col('unit_price').cast('double')) \
.withColumn('total_amount', col('total_amount').cast('double')) \
.dropna(subset=['sale_id','drug_name','region'])

df_clean.show()

In [0]:
#  Cell 5: Apply filters from widgets 
if region != 'All':
    df_clean = df_clean.filter(col('region') == region)

df_clean = df_clean.filter(
(col('sale_date') >= start_date) & (col('sale_date') <= end_date)
)
log_counter(df_clean, f'After filters (region={region})')
df_clean.show()

In [0]:
#  Cell 6: Add derived columns 
df_clean = df_clean \
.withColumn('revenue_band',
when(col('total_amount') >= 2000, 'High')
.when(col('total_amount') >= 1500, 'Medium')
.otherwise('Low')
) \
.withColumn('margin_estimate',
round(col('total_amount') * 0.35, 2) # 35% margin assumption
)
df_clean.createOrReplaceTempView('clean_sales')

df_clean.show()

In [0]:
%sql
SELECT revenue_band, COUNT(*) AS records FROM clean_sales GROUP BY revenue_band;

In [0]:
dbutils.notebook.exit(str(df_clean.count()))